
## Trying to make a multilingual/scandinavian NER model using transformers.

## Importing libraries

Mostly using HuggingFace + PyTorch things here.

In [ ]:
#     !pip install transformers datasets seqeval torch accelerate -q

## Loading dataset

WikiANN dataset already has NER labels so easier to work with.

In [ ]:
import os
import numpy as np
import torch
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from datasets import load_dataset, Dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score



## Tokenization


In [ ]:
#Training was super slow on cpu
# was getting shape mismatch before
# copied the general idea from HF docs and adjusted it
# memory crash happened before so reduced batch size

# using Trainer API because manual loop was too long
# can probably clean this code later

#Model Details: 
MODEL_NAME = 'bert-base-multilingual-cased'

# Languages we have to Train 
LANGUAGES = {
    'da': 'Danish',
    'sv': 'Swedish',
    'no': 'Norwegian',
}

#NER label scheme (same 7 labels as the English baseline) 
LABEL_LIST  = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']
LABEL_TO_ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

#raining hyperparameters (exact copy from the English baseline) 
LEARNING_RATE      = 2e-5
EPOCHS             = 3
TRAIN_BATCH_SIZE   = 8
EVAL_BATCH_SIZE    = 16
WEIGHT_DECAY       = 0.01
WARMUP_STEPS       = 500
LOGGING_STEPS      = 50
SEED               = 42
FP16               = True# Reminder:: Set 'False' if the machine doesn't have a GPU
MAX_LENGTH         = 512

print(f'Model          : {MODEL_NAME}')
print(f'Languages      : {list(LANGUAGES.values())}')
print(f'Label scheme   : {LABEL_LIST}')
print(f'Hyperparameters: lr={LEARNING_RATE}, epochs={EPOCHS}, batch={TRAIN_BATCH_SIZE}, seed={SEED}')


## Label alignment

Need to align labels correctly otherwise training will break.

In [ ]:
#Loading all three WikiANN datasets from HuggingFace 
# training was super slow on cpu
raw_datasets = {}
for lang_code, lang_name in LANGUAGES.items():
    print(f'Loading WikiANN for {lang_name} ({lang_code})...')
    raw_datasets[lang_code] = load_dataset('wikiann', lang_code)

# Now we Build remapping: WikiANN integer IDs → our LABEL_LIST order 
# WikiANN has the same 7 labels but in a DIFFERENT order:
#   WikiANN: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC
#   Ours   : O, B-PER, I-PER, B-LOC, I-LOC, B-ORG, I-ORG
# So we remap WikiANN integer IDs to our label order.
wikiann_label_names = raw_datasets['da']['train'].features['ner_tags'].feature.names
print(f'\nWikiANN label order: {wikiann_label_names}')
print(f'Our label order   : {LABEL_LIST}')

WIKIANN_TO_OURS = {}
for wiki_id, label_name in enumerate(wikiann_label_names):
    WIKIANN_TO_OURS[wiki_id] = LABEL_TO_ID[label_name]
print(f'Remap table: {WIKIANN_TO_OURS}')
#Print split sizes 
print()
for lang_code, lang_name in LANGUAGES.items():
    ds = raw_datasets[lang_code]
    print(f'{lang_name:>12} — train: {len(ds["train"]):>6,}  '
          f'val: {len(ds["validation"]):>6,}  '
          f'test: {len(ds["test"]):>6,}')



## Training config

Using Trainer API because manual training loop was too long.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer Loaded: {MODEL_NAME}')
print(f'VocabularySize : {tokenizer.vocab_size:,}')

## Model training

Hopefully the model learns decent entity patterns here.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
        padding=False,
    )
    all_labels = []
    for i, tag_ids in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev_word_idx = [], None
        for word_idx in word_ids:
            if word_idx is None:
                #special token ([CLS], [SEP], padding)
                aligned.append(-100)
            elif word_idx != prev_word_idx:
                #first subword of a new word → use the real label
                #remap WikiANN ID → our model's label ID
                aligned.append(WIKIANN_TO_OURS[tag_ids[word_idx]])
            else:
                        # Continuation subword → ignore
                aligned.append(-100)
            prev_word_idx = word_idx
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized


# Now its time to use tokenization  to every other language and split 
tokenized_datasets = {}
for lang_code, lang_name in LANGUAGES.items():
    print(f'Tokenizing {lang_name}...')
    tokenized_datasets[lang_code] = {}
    for split in ['train', 'validation', 'test']:
        tokenized_datasets[lang_code][split] = raw_datasets[lang_code][split].map(
            tokenize_and_align_labels,
            batched=True,
            remove_columns=raw_datasets[lang_code][split].column_names,
        )
    tr = len(tokenized_datasets[lang_code]['train'])
    va = len(tokenized_datasets[lang_code]['validation'])
    te = len(tokenized_datasets[lang_code]['test'])
    print(f'  {lang_name}: train={tr}  val={va}  test={te}')





## Evaluation


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU name    : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Predictions



In [ ]:
# Everything going well so far..
def compute_metrics(eval_preds):
    logits, true_labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_strs, pred_strs = [], []
    for pred_seq, true_seq in zip(predictions, true_labels):
        true_s, pred_s = [], []
        for p, t in zip(pred_seq, true_seq):
            if t == -100:
                continue
            true_s.append(ID_TO_LABEL[t])
            pred_s.append(ID_TO_LABEL[p])
        true_strs.append(true_s)
        pred_strs.append(pred_s)
    return {
        'precision': precision_score(true_strs, pred_strs),
        'recall':    recall_score(true_strs, pred_strs),
        'f1':        f1_score(true_strs, pred_strs),
    }

## Saving everything


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

#We'll store each language's trainer and results here
trainers = {}
results  = {}

for lang_code, lang_name in LANGUAGES.items():
    print(f'\n{"="*60}')
    print(f'  TRAINING: {lang_name.upper()} ({lang_code})')
    print(f'{"="*60}\n')

    #1. Fresh model from pretrained mBERT 
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_LIST),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        ignore_mismatched_sizes=True,
    )
    model = model.to(device)

        #2. Training arguments (same as English baseline) 
    output_dir = f'./ner_model_output_{lang_code}'
    training_args = TrainingArguments(
        output_dir=output_dir,
        fp16=FP16,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        logging_steps=LOGGING_STEPS,
        report_to='none',
        seed=SEED,
    )

        # 3. Build Trainer 
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets[lang_code]['train'],
        eval_dataset=tokenized_datasets[lang_code]['validation'],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

            # 4. Train 
    train_result = trainer.train()
    print(f'\n{lang_name} training loss: {train_result.training_loss:.4f}')

    #5. Save trainer for later use 
    trainers[lang_code] = trainer
    results[lang_code]  = train_result

print(f'\n{"="*60}')
print('ALL TRAINING COMPLETE')
print(f'{"="*60}')



## Dev Set Evaluation


In [ ]:
for lang_code, lang_name in LANGUAGES.items():
    print(f'\n{"="*60}')
    print(f'  DEV EVALUATION: {lang_name.upper()} ({lang_code})')
    print(f'{"="*60}\n')

    trainer = trainers[lang_code]
    dev_preds_raw = trainer.predict(tokenized_datasets[lang_code]['validation'])
    dev_preds = np.argmax(dev_preds_raw.predictions, axis=-1)
    dev_true  = dev_preds_raw.label_ids

#Convert integer IDs back to string labels, skipping -100 padding
    true_strs, pred_strs = [], []
    for pred_seq, true_seq in zip(dev_preds, dev_true):
        t, p = [], []
        for pred_id, true_id in zip(pred_seq, true_seq):
            if true_id == -100:
                continue
            t.append(ID_TO_LABEL[true_id])
            p.append(ID_TO_LABEL[pred_id])
        true_strs.append(t)
        pred_strs.append(p)

    print(f'F1        : {f1_score(true_strs, pred_strs):.4f}')
    print(f'Precision : {precision_score(true_strs, pred_strs):.4f}')
    print(f'Recall    : {recall_score(true_strs, pred_strs):.4f}')
    print(classification_report(true_strs, pred_strs))


## Generate Test Predictions

In [ ]:
for lang_code, lang_name in LANGUAGES.items():
    print(f'\nGenerating test predictions for {lang_name}...')

    trainer  = trainers[lang_code]
    test_ds  = tokenized_datasets[lang_code]['test']
    test_raw = raw_datasets[lang_code]['test']

    #Get model predictions on the test set 
    test_preds_raw = trainer.predict(test_ds)
    test_preds     = np.argmax(test_preds_raw.predictions, axis=-1)

        #Re-tokenize to get word_ids for subword→word mapping 
    test_tokenized_for_ids = tokenizer(
        [ex['tokens'] for ex in test_raw],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
        padding=False,
    )

            # Map  subword predictions back to word-level labels 
    output_predictions = []
    for i, pred_seq in enumerate(test_preds):
        word_ids = test_tokenized_for_ids.word_ids(batch_index=i)
        sentence_preds, prev_word_idx = [], None
        for j, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx != prev_word_idx:
                #First subword of this word → take its prediction
                sentence_preds.append(ID_TO_LABEL[pred_seq[j]])
            prev_word_idx = word_idx
        output_predictions.append(sentence_preds)

    # Write to .iob2 file 
    output_path = f'./test_predictions_{lang_code}.iob2'
    with open(output_path, 'w', encoding='utf-8') as f:
        for example, preds in zip(test_raw, output_predictions):
            for token, pred in zip(example['tokens'], preds):
                f.write(f'{token}\t{pred}\n')
            f.write('\n')

    print(f'  Saved → {output_path}  ({len(output_predictions)} sentences)')



## Save Models & Tokenizers
We save each fine-tuned model and the tokenizer to disk so we don't have to retrain.
Each language gets its own folder.


In [ ]:
for lang_code, lang_name in LANGUAGES.items():
    output_dir = f'./ner_model_output_{lang_code}'
    trainers[lang_code].save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f'{lang_name} model and tokenizer saved to: {output_dir}/')

#To reload later:
#model = AutoModelForTokenClassification.from_pretrained('./ner_model_output_da')
#tokenizer = AutoTokenizer.from_pretrained('./ner_model_output_da')

## Results Summary Table


In [ ]:
print(f'{"Language":<12} {"F1":>8} {"Precision":>10} {"Recall":>8} {"Train Loss":>12}')
print('-' * 54)

for lang_code, lang_name in LANGUAGES.items():
    trainer = trainers[lang_code]

    #Re-run quick eval to get metrics
    dev_preds_raw = trainer.predict(tokenized_datasets[lang_code]['validation'])
    dev_preds = np.argmax(dev_preds_raw.predictions, axis=-1)
    dev_true  = dev_preds_raw.label_ids

    true_strs, pred_strs = [], []
    for pred_seq, true_seq in zip(dev_preds, dev_true):
        t, p = [], []
        for pred_id, true_id in zip(pred_seq, true_seq):
            if true_id == -100:
                continue
            t.append(ID_TO_LABEL[true_id])
            p.append(ID_TO_LABEL[pred_id])
        true_strs.append(t)
        pred_strs.append(p)

    f1  = f1_score(true_strs, pred_strs)
    pre = precision_score(true_strs, pred_strs)
    rec = recall_score(true_strs, pred_strs)
    loss = results[lang_code].training_loss

    print(f'{lang_name:<12} {f1:>8.4f} {pre:>10.4f} {rec:>8.4f} {loss:>12.4f}')

